# Text Classification with Bag-of-Words and Keras MLP

This notebook demonstrates how to perform text classification using the Bag-of-Words (BOW) approach for feature extraction and a simple Keras Multi-Layer Perceptron (MLP) for classification.

We will:
- Load and explore a sample text dataset
- Preprocess the text data
- Convert text to numeric features using BOW
- Build and train a Keras MLP model
- Evaluate the model's performance

## 1. Load and Explore Text Data

We will use a small sample dataset for demonstration. Each text is labeled as either positive (1) or negative (0). Let's look at a few examples.

### Example Texts and Labels

| Text                              | Label |
|-----------------------------------|-------|
| I love this movie!                |   1   |
| This film was terrible.           |   0   |
| What a fantastic experience.      |   1   |
| I did not enjoy the plot.         |   0   |
| Absolutely wonderful performance. |   1   |

We will use a similar structure in our code.

In [9]:
# Sample text data and labels
texts = [
    "I love this movie!",
    "This film was terrible.",
    "What a fantastic experience.",
    "I did not enjoy the plot.",
    "Absolutely wonderful performance."
]
labels = [1, 0, 1, 0, 1]

for text, label in zip(texts, labels):
    print(f"Text: {text:40} Label: {label}")

Text: I love this movie!                       Label: 1
Text: This film was terrible.                  Label: 0
Text: What a fantastic experience.             Label: 1
Text: I did not enjoy the plot.                Label: 0
Text: Absolutely wonderful performance.        Label: 1


## 2. Text Preprocessing

Before vectorizing, we preprocess the text by lowercasing, removing punctuation, and tokenizing. This helps standardize the input for the vectorizer.

### Example: Preprocessing a Sentence

Original: `I love this movie!`

- Lowercased: `i love this movie!`
- Remove punctuation: `i love this movie`
- Tokenized: `["i", "love", "this", "movie"]`

In [10]:
import re

def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Tokenize (split by whitespace)
    tokens = text.split()
    return tokens

# Example
example = texts[0]
print("Original:", example)
print("Processed:", preprocess(example))

Original: I love this movie!
Processed: ['i', 'love', 'this', 'movie']


## 3. Bag-of-Words (BOW) Vectorization

Bag-of-Words is a method to convert text into fixed-length numeric vectors by counting word occurrences. Each unique word in the dataset becomes a feature (column) in the vectorized representation.

### How BOW Works

Suppose our dataset has the following sentences:
- "I love this movie"
- "This movie was terrible"

The vocabulary is: `[I, love, this, movie, was, terrible]`

Each sentence is represented as a vector of word counts:
- [1, 1, 1, 1, 0, 0]  (for "I love this movie")
- [0, 0, 1, 1, 1, 1]  (for "This movie was terrible")

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

# Example with two sentences
demo_texts = ["I love this movie", "This movie was terrible"]
vectorizer = CountVectorizer()
X_demo = vectorizer.fit_transform(demo_texts)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("BOW Vectors:\n", X_demo.toarray())

Vocabulary: ['love' 'movie' 'terrible' 'this' 'was']
BOW Vectors:
 [[1 1 0 1 0]
 [0 1 1 1 1]]


## 4. Vectorize the Dataset

Now, let's apply the BOW vectorizer to our full dataset. This will convert each text into a numeric feature vector suitable for input to a machine learning model.

In [12]:
# Vectorize the full dataset
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

print("Feature names:", vectorizer.get_feature_names_out())
print("Feature matrix shape:", X.shape)
print("First vector:", X.toarray()[0])

Feature names: ['absolutely' 'did' 'enjoy' 'experience' 'fantastic' 'film' 'love' 'movie'
 'not' 'performance' 'plot' 'terrible' 'the' 'this' 'was' 'what'
 'wonderful']
Feature matrix shape: (5, 17)
First vector: [0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0]


## 5. Build a Simple Keras MLP Model

We will use a simple Multi-Layer Perceptron (MLP) for classification. The input layer size matches the number of BOW features. The output layer uses a sigmoid activation for binary classification.

### Model Architecture
- Input layer: size = number of BOW features
- Hidden layer: Dense, e.g., 8 units, ReLU activation
- Output layer: 1 unit, sigmoid activation (for binary classification)

In [13]:
from tensorflow import keras
from tensorflow.keras import layers

In [14]:
input_dim = X.shape[1]
input_dim

17

In [15]:
model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 8)              │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 153 (612.00 B)

 Trainable params: 153 (612.00 B)

 Non-trainable params: 0 (0.00 B)

## 6. Train the Model

We will compile the model with binary crossentropy loss and the Adam optimizer, then train it on our data.

In [16]:
import numpy as np

# Convert labels to numpy array
labels_np = np.array(labels)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X.toarray(), labels_np, epochs=20, verbose=1)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 595ms/step - accuracy: 0.4000 - loss: 0.7100
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4000 - loss: 0.7080
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4000 - loss: 0.7060
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4000 - loss: 0.7039
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4000 - loss: 0.7020
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.6000 - loss: 0.7000
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6000 - loss: 0.6980
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6000 - loss: 0.6960
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6000 - loss: 0.6941
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6000 - loss: 0.6922
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6000 - loss: 0.6903
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6000 - loss: 0.6884


## 7. Evaluate Model Performance

Let's evaluate the model's accuracy and see some example predictions.

In [17]:
# Evaluate accuracy
loss, acc = model.evaluate(X.toarray(), labels_np, verbose=0)
print(f"Accuracy: {acc:.2f}")

# Example predictions
preds = model.predict(X.toarray())
print("Predictions:", np.round(preds.ravel()))
print("True labels:", labels_np)

Accuracy: 0.60
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Predictions: [0. 0. 1. 1. 1.]
True labels: [1 0 1 0 1]


#### Extending this to large datasets

- IMDB Movie Reviews

In [18]:
import pandas as pd

# Load IMDB dataset from the given path (outside workspace)
df = pd.read_csv(r'D:\Makesh\Working\AI\RPS\Day04\IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [19]:
df.shape

(50000, 2)

### Bag-of-Words and CountVectorizer on IMDB Dataset

We will now:
- Use CountVectorizer to convert the IMDB reviews into BOW features
- Train a simple Keras MLP for sentiment classification

Let's start by checking the data format and preparing the text and label arrays.

In [20]:
# Prepare text and labels from IMDB DataFrame
txts = df['review'].values
labels = (df['sentiment'] == 'positive').astype(int).values

print(f"Number of samples: {len(txts)}")
print(f"Example review: {txts[0][:100]}...")
print(f"Example label: {labels[0]}")

Number of samples: 50000
Example review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. The...
Example label: 1


In [21]:
from sklearn.feature_extraction.text import CountVectorizer

# Use a limited vocabulary for speed in demo (e.g., top 5000 words)
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_bow = vectorizer.fit_transform(txts)

print("BOW feature matrix shape:", X_bow.shape)
print("First review vector (truncated):", X_bow.toarray()[0][:20])

BOW feature matrix shape: (50000, 5000)
First review vector (truncated): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


### Train/Test Split

Let's split the data into training and test sets for model evaluation.

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_bow, labels, test_size=0.2, random_state=42, stratify=labels)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (40000, 5000), Test shape: (10000, 5000)


### Keras MLP for IMDB Sentiment Classification

We will now build and train a simple Keras MLP using the BOW features.

In [28]:
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X_train.shape[1]

mlp = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
mlp.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 32)             │       160,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,065 (625.25 KB)

 Trainable params: 160,065 (625.25 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
# Train the MLP model
history = mlp.fit(
    X_train.toarray(), y_train,
    validation_data=(X_test.toarray(), y_test),
    epochs=5, batch_size=128
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9271 - loss: 0.1863 - val_accuracy: 0.8755 - val_loss: 0.3201
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9354 - loss: 0.1665 - val_accuracy: 0.8764 - val_loss: 0.3364
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9456 - loss: 0.1445 - val_accuracy: 0.8738 - val_loss: 0.3435
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9557 - loss: 0.1216 - val_accuracy: 0.8735 - val_loss: 0.3630
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9675 - loss: 0.0989 - val_accuracy: 0.8728 - val_loss: 0.3875


In [30]:
# Evaluate the model
loss, acc = mlp.evaluate(X_test.toarray(), y_test, verbose=0)
print(f"Test Accuracy: {acc:.2f}")

Test Accuracy: 0.88


### Discussion: High Accuracy with Bag-of-Words

With just a simple Bag-of-Words (CountVectorizer) and a basic Keras MLP, we achieved around **88% accuracy** on the IMDB sentiment classification task.

**Why does this work well?**
- The IMDB dataset is large and well-labeled, so even simple models can learn strong patterns.
- Many sentiment words (e.g., "great", "terrible", "love", "boring") are highly predictive and captured by BOW.
- The MLP can learn non-linear combinations of these word features.

**Limitations:**
- BOW ignores word order and context.
- More advanced models (e.g., LSTM, transformers) can do even better, especially on more nuanced tasks.

You can now experiment with more complex models or try additional preprocessing for further improvements!

## Background: What are Word Vectors?

Word vectors (or word embeddings) are dense, low-dimensional representations of words, where similar words have similar vector representations. Unlike Bag-of-Words, which uses sparse and high-dimensional vectors, word embeddings capture semantic meaning and relationships between words.

**Key points:**
- Each word is mapped to a real-valued vector (e.g., 100 or 300 dimensions).
- Vectors are learned from large text corpora (e.g., Word2Vec, GloVe, FastText).
- Words with similar meanings are close together in the vector space.
- Embeddings can be used as input to neural networks for NLP tasks.

**Example:**
- The vectors for "king" and "queen" will be close, and the difference between "man" and "woman" is similar to the difference between "king" and "queen".

In the next steps, you can use pre-trained word vectors or train your own embeddings within a neural network for better text representation.

### Why Learn Word2Vec (or Embeddings) When Count Vectors Work Well?

While Bag-of-Words (BOW) and CountVectorizer can achieve high accuracy on some tasks, there are important reasons to learn and use word embeddings like Word2Vec:

**1. Semantic Meaning:**
- BOW treats every word as independent; it cannot capture relationships or similarities between words (e.g., "good" and "great" are unrelated in BOW).
- Word2Vec places similar words close together in vector space, capturing meaning and context.

**2. Dimensionality Reduction:**
- BOW creates very high-dimensional, sparse vectors (one dimension per word).
- Embeddings are dense and low-dimensional (e.g., 100-300 dimensions), making models faster and less prone to overfitting.

**3. Generalization and Transfer:**
- Embeddings can generalize better to unseen words or phrases and can be transferred to new tasks (transfer learning).
- Pre-trained embeddings (e.g., GloVe, FastText) bring in knowledge from huge text corpora.

**4. Context and Polysemy:**
- Advanced embeddings (e.g., contextual embeddings like BERT) can capture word meaning in context and handle words with multiple meanings.

**5. Limitations of BOW:**
- BOW ignores word order and context, which can be critical for nuanced understanding (e.g., "not good" vs. "good").
- Embeddings can be used in models (like RNNs, CNNs, Transformers) that consider word order and context.

**Summary:**
- BOW is a strong baseline for simple tasks and large, clean datasets.
- For more complex language understanding, generalization, and transfer, word embeddings are essential.
- Learning Word2Vec and embeddings opens the door to more powerful and flexible NLP models.